In [1]:
import re
import os.path as osp
import pandas as pd
from configuration_template import Config
from data_prep.data_utils import build_patch_image_with_target_highlight

config = Config()


def extract_pattern(response_sentence):
    """
    Extract the response pattern (e.g., 'left', 'middle', 'right') from the model output.
    """
    pattern = r"\b(left|middle|right)\b"
    match = re.search(pattern, response_sentence.lower())
    return match.group(0) if match else None


def utterances_to_dialogue_string(utterances):
    lines = [f'{intl}: {utt}' for intl, utt, _ in utterances]
    return '\n'.join(lines)

# define prompts
TOP_INSTRUCTION = 'In this image you can see three color patches. In the following dialogue, the speaker I will describe exactly one of the patches. Please indicate to me whether he refers to the left, middle or right patch.'
BOTTOM_PROMPT = 'Is it the left, middle or right patch?'


# settings
img_kwargs = {
    "patch_size": config.patch_size*3,
    "patch_padding": config.grid_padding,
    "patch_pad_color": config.grid_pad_color,
}


# load data
data_df = pd.read_json(osp.join(config.data_dir, "color_patch_data.json"))

model_version: Qwen2
model_type: Qwen


In [2]:
data_df

,identifier,gameid,roundNum,condition,success,hls_values,s_order_t_distr1_distr2,l_order_t_distr1_distr2,conversation,n_utterances
0,1124-1:1,1124-1,1,close,False,"[[226, 50, 81], [283, 50, 87], [248, 50, 92]]","[0, 1, 2]","[1, 2, 0]","[[speaker, The darker blue one, None]]",1
1,1124-1:2,1124-1,2,far,True,"[[249, 50, 25], [129, 50, 66], [54, 50, 53]]","[0, 1, 2]","[1, 0, 2]","[[speaker, purple, None]]",1
2,1124-1:3,1124-1,3,close,False,"[[301, 50, 57], [302, 50, 86], [291, 50, 59]]","[1, 0, 2]","[1, 0, 2]","[[speaker, Medium pink, None], [speaker, the m...",2
3,1124-1:4,1124-1,4,split,False,"[[87, 50, 66], [85, 50, 21], [299, 50, 29]]","[0, 1, 2]","[1, 0, 2]","[[speaker, lime, None]]",1
4,1124-1:5,1124-1,5,far,True,"[[156, 50, 84], [57, 50, 64], [239, 50, 87]]","[0, 2, 1]","[2, 0, 1]","[[speaker, Mint green., None]]",1
...,...,...,...,...,...,...,...,...,...,...
47036,8452-5:46,8452-5,46,close,True,"[[191, 50, 2], [197, 50, 13], [140, 50, 12]]","[1, 2, 0]","[1, 0, 2]","[[speaker, grey, None]]",1
47037,8452-5:47,8452-5,47,split,True,"[[69, 50, 94], [85, 50, 78], [179, 50, 41]]","[2, 1, 0]","[2, 0, 1]","[[speaker, yellow, None]]",1
47038,8452-5:48,8452-5,48,split,True,"[[52, 50, 44], [35, 50, 24], [359, 50, 30]]","[1, 2, 0]","[2, 1, 0]","[[speaker, yellow, None]]",1
47039,8452-5:49,8452-5,49,far,True,"[[284, 50, 24], [174, 50, 35], [319, 50, 1]]","[1, 2, 0]","[2, 0, 1]","[[speaker, purple, None]]",1


In [3]:
data_df_grid = pd.read_json(osp.join(config.data_dir, "color_grid_data.json"))
data_df_grid

,gameid,game_idx,roundNum,round_id,condition,success,utterances,n_utterances,objs,target,speaker_order,listener_order,listener_clicked
0,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_1,FAR,True,"[[speaker, green top left corner, 1528144087280]]",1,"[{'shapes': [{'color': [322, 21, 50]}, {'color...",1,"[2, 1, 0]","[1, 0, 2]",0
1,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,2,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_2,SPLIT,True,"[[speaker, pink in the bottom left, 1528144112...",1,"[{'shapes': [{'color': [55, 5, 50]}, {'color':...",2,"[2, 1, 0]","[0, 2, 1]",1
2,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,3,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_3,SPLIT,False,"[[speaker, dark orange top right corner, 15281...",1,"[{'shapes': [{'color': [206, 56, 50]}, {'color...",0,"[0, 1, 2]","[1, 2, 0]",1
3,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_4,FAR,True,"[[speaker, green in the middle square top row,...",1,"[{'shapes': [{'color': [294, 7, 50]}, {'color'...",0,"[1, 2, 0]","[2, 1, 0]",2
4,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619,1,5,4147-9d9582f2-28d0-431f-b6ba-35fd1be76619_5,SPLIT,True,"[[speaker, top right corner red, 1528144188447]]",1,"[{'shapes': [{'color': [227, 76, 50]}, {'color...",1,"[0, 2, 1]","[1, 2, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10920,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,56,6068-c8135454-2875-41d9-918c-60c34fc9460c_56,CLOSE,True,"[[speaker, top right not darkest not lightest ...",1,"[{'shapes': [{'color': [121, 13, 50]}, {'color...",1,"[0, 1, 2]","[2, 0, 1]",2
10921,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,57,6068-c8135454-2875-41d9-918c-60c34fc9460c_57,FAR,True,"[[speaker, Top middle green, 1526435245005]]",1,"[{'shapes': [{'color': [195, 34, 50]}, {'color...",1,"[2, 1, 0]","[1, 2, 0]",0
10922,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,58,6068-c8135454-2875-41d9-918c-60c34fc9460c_58,FAR,True,"[[speaker, middle right pink, 1526435266994]]",1,"[{'shapes': [{'color': [187, 64, 50]}, {'color...",2,"[0, 1, 2]","[2, 0, 1]",0
10923,6068-c8135454-2875-41d9-918c-60c34fc9460c,198,59,6068-c8135454-2875-41d9-918c-60c34fc9460c_59,CLOSE,True,"[[speaker, Bottom middle is the boldest yellow...",1,"[{'shapes': [{'color': [48, 19, 50]}, {'color'...",0,"[1, 0, 2]","[1, 2, 0]",2
